# E. coli Model Comparison: iML1515 and iJO1366

This notebook compares two genome-scale metabolic models of *Escherichia coli*: iML1515 and iJO1366.

Both models represent *E. coli* K-12 MG1655, but they differ in reconstruction scope, number of reactions, metabolites, and genes.

The goal is to evaluate whether two different reconstructions produce similar or different Flux Balance Analysis predictions under the same medium conditions.

The analysis includes:

- loading and inspecting both models
- checking required exchange reactions
- applying the same medium conditions to both models
- running FBA for each model-condition combination
- comparing objective values
- comparing active fluxes
- comparing blocked reactions
- exporting summary tables and figures

## 1. Imports

The required Python libraries are imported in this section.

COBRApy is used to load and analyze genome-scale metabolic models, pandas is used for tabular data handling, and matplotlib is used for generating comparison plots.

In [18]:
from pathlib import Path

import cobra
from cobra.io import load_model
from cobra.flux_analysis import find_blocked_reactions

import pandas as pd
import matplotlib.pyplot as plt

## 2. Output folders

This section defines the output directories used to store generated tables and figures.

All CSV files are saved in `outputs/tables`, while all PNG figures are saved in `outputs/figures`.

In [19]:
output_dir = Path("outputs")
tables_dir = output_dir / "tables"
figures_dir = output_dir / "figures"

for folder in [output_dir, tables_dir, figures_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print("Output folders are ready.")

Output folders are ready.


## 3. Load E. coli genome-scale models

Two genome-scale metabolic models of *Escherichia coli* are loaded using COBRApy.

The models are:

- iML1515
- iJO1366

Both models are used for Flux Balance Analysis under the same medium conditions.

In [20]:
models = {
    "iML1515": load_model("iML1515"),
    "iJO1366": load_model("iJO1366")
}

for model_name, model_obj in models.items():
    print("=" * 60)
    print("Model:", model_name)
    print("Model ID:", model_obj.id)
    print("Number of reactions:", len(model_obj.reactions))
    print("Number of metabolites:", len(model_obj.metabolites))
    print("Number of genes:", len(model_obj.genes))
    print("Objective:", model_obj.objective.expression)

Model: iML1515
Model ID: iML1515
Number of reactions: 2712
Number of metabolites: 1877
Number of genes: 1516
Objective: 1.0*BIOMASS_Ec_iML1515_core_75p37M - 1.0*BIOMASS_Ec_iML1515_core_75p37M_reverse_35685
Model: iJO1366
Model ID: iJO1366
Number of reactions: 2583
Number of metabolites: 1805
Number of genes: 1367
Objective: 1.0*BIOMASS_Ec_iJO1366_core_53p95M - 1.0*BIOMASS_Ec_iJO1366_core_53p95M_reverse_5c8b1


## 4. Initial model comparison table

A first comparison table is created to summarize the basic structural characteristics of the two models.

This includes the number of reactions, metabolites, genes, and the objective reaction used for FBA.

In [21]:
model_overview_rows = []

for model_name, model_obj in models.items():
    model_overview_rows.append({
        "model_name": model_name,
        "model_id": model_obj.id,
        "num_reactions": len(model_obj.reactions),
        "num_metabolites": len(model_obj.metabolites),
        "num_genes": len(model_obj.genes),
        "objective": str(model_obj.objective.expression)
    })

model_overview_df = pd.DataFrame(model_overview_rows)

model_overview_df

,model_name,model_id,num_reactions,num_metabolites,num_genes,objective
0,iML1515,iML1515,2712,1877,1516,1.0*BIOMASS_Ec_iML1515_core_75p37M - 1.0*BIOMA...
1,iJO1366,iJO1366,2583,1805,1367,1.0*BIOMASS_Ec_iJO1366_core_53p95M - 1.0*BIOMA...


In [22]:
model_overview_path = tables_dir / "ecoli_model_overview_iML1515_iJO1366.csv"

model_overview_df.to_csv(model_overview_path, index=False)

print("Saved:", model_overview_path)

Saved: outputs\tables\ecoli_model_overview_iML1515_iJO1366.csv


## 5. Required exchange reactions check

Before applying the same medium conditions to both models, the required exchange reactions are checked.

This confirms that both iML1515 and iJO1366 contain the exchange reactions needed for the selected carbon sources, oxygen, and essential inorganic nutrients.

In [23]:
required_exchanges = [
    "EX_glc__D_e",   # D-glucose
    "EX_fru_e",      # fructose
    "EX_gal_e",      # galactose
    "EX_glyc_e",     # glycerol
    "EX_succ_e",     # succinate
    "EX_ac_e",       # acetate
    "EX_o2_e",       # oxygen
    "EX_nh4_e",      # ammonium
    "EX_pi_e",       # phosphate
    "EX_h2o_e",      # water
    "EX_h_e",        # proton
    "EX_k_e",        # potassium
    "EX_na1_e",      # sodium
    "EX_so4_e",      # sulfate
    "EX_cl_e",       # chloride
    "EX_mg2_e",      # magnesium
    "EX_ca2_e",      # calcium
    "EX_fe2_e",      # iron II
    "EX_fe3_e"       # iron III
]

for model_name, model_obj in models.items():
    print("=" * 60)
    print("Model:", model_name)

    for reaction_id in required_exchanges:
        if reaction_id in model_obj.reactions:
            print(reaction_id, "FOUND")
        else:
            print(reaction_id, "NOT FOUND")

Model: iML1515
EX_glc__D_e FOUND
EX_fru_e FOUND
EX_gal_e FOUND
EX_glyc_e FOUND
EX_succ_e FOUND
EX_ac_e FOUND
EX_o2_e FOUND
EX_nh4_e FOUND
EX_pi_e FOUND
EX_h2o_e FOUND
EX_h_e FOUND
EX_k_e FOUND
EX_na1_e FOUND
EX_so4_e FOUND
EX_cl_e FOUND
EX_mg2_e FOUND
EX_ca2_e FOUND
EX_fe2_e FOUND
EX_fe3_e FOUND
Model: iJO1366
EX_glc__D_e FOUND
EX_fru_e FOUND
EX_gal_e FOUND
EX_glyc_e FOUND
EX_succ_e FOUND
EX_ac_e FOUND
EX_o2_e FOUND
EX_nh4_e FOUND
EX_pi_e FOUND
EX_h2o_e FOUND
EX_h_e FOUND
EX_k_e FOUND
EX_na1_e FOUND
EX_so4_e FOUND
EX_cl_e FOUND
EX_mg2_e FOUND
EX_ca2_e FOUND
EX_fe2_e FOUND
EX_fe3_e FOUND


In [24]:
exchange_check_rows = []

for model_name, model_obj in models.items():
    for reaction_id in required_exchanges:
        exchange_check_rows.append({
            "model_name": model_name,
            "reaction_id": reaction_id,
            "found": reaction_id in model_obj.reactions
        })

exchange_check_df = pd.DataFrame(exchange_check_rows)

exchange_check_df

,model_name,reaction_id,found
0,iML1515,EX_glc__D_e,True
1,iML1515,EX_fru_e,True
2,iML1515,EX_gal_e,True
3,iML1515,EX_glyc_e,True
4,iML1515,EX_succ_e,True
5,iML1515,EX_ac_e,True
6,iML1515,EX_o2_e,True
7,iML1515,EX_nh4_e,True
8,iML1515,EX_pi_e,True
9,iML1515,EX_h2o_e,True


In [25]:
exchange_check_path = tables_dir / "ecoli_model_required_exchange_check.csv"

exchange_check_df.to_csv(exchange_check_path, index=False)

print("Saved:", exchange_check_path)

Saved: outputs\tables\ecoli_model_required_exchange_check.csv


## 6. Medium definition helper function

This helper function creates a copy of a model and modifies its medium.

The default medium of each model is used as a base so that essential nutrients and ions are preserved. Only the carbon source and oxygen availability are changed.

This allows both models to be tested under equivalent medium conditions.

In [26]:
def set_medium_from_default(base_model, carbon_source, oxygen=True, carbon_uptake=10):
    """
    Set medium condition using the model's default medium as a base.
    This keeps the essential nutrients already defined in the model
    and changes only the carbon source and oxygen availability.
    """

    model_condition = base_model.copy()
    medium = model_condition.medium.copy()

    carbon_sources_to_remove = [
        "EX_glc__D_e",
        "EX_fru_e",
        "EX_gal_e",
        "EX_glyc_e",
        "EX_succ_e",
        "EX_ac_e",
        "EX_lac__D_e",
        "EX_lac__L_e",
        "EX_pyr_e",
        "EX_man_e"
    ]

    for source in carbon_sources_to_remove:
        if source in medium:
            medium[source] = 0

    medium[carbon_source] = carbon_uptake

    if "EX_o2_e" in model_condition.reactions:
        if oxygen:
            medium["EX_o2_e"] = 1000
        else:
            medium["EX_o2_e"] = 0

    model_condition.medium = medium

    return model_condition

In [27]:
for model_name, model_obj in models.items():
    test_model = set_medium_from_default(
        base_model=model_obj,
        carbon_source="EX_glc__D_e",
        oxygen=True,
        carbon_uptake=10
    )

    test_solution = test_model.optimize()

    print("=" * 60)
    print("Model:", model_name)
    print("Test condition: glucose_aerobic")
    print("Solution status:", test_solution.status)
    print("Objective value:", test_solution.objective_value)

Model: iML1515
Test condition: glucose_aerobic
Solution status: optimal
Objective value: 0.8769972144269811
Model: iJO1366
Test condition: glucose_aerobic
Solution status: optimal
Objective value: 0.9823718127269785


## 7. Define shared medium conditions

The same medium conditions are defined for both E. coli models.

Using identical conditions allows the comparison to focus on differences between the two model reconstructions rather than differences in medium setup.

The comparison includes six aerobic carbon source conditions and one anaerobic glucose condition.

In [28]:
conditions = {
    "glucose_aerobic": {
        "carbon_source": "EX_glc__D_e",
        "oxygen": True,
        "carbon_uptake": 10
    },
    "fructose_aerobic": {
        "carbon_source": "EX_fru_e",
        "oxygen": True,
        "carbon_uptake": 10
    },
    "galactose_aerobic": {
        "carbon_source": "EX_gal_e",
        "oxygen": True,
        "carbon_uptake": 10
    },
    "glycerol_aerobic": {
        "carbon_source": "EX_glyc_e",
        "oxygen": True,
        "carbon_uptake": 10
    },
    "succinate_aerobic": {
        "carbon_source": "EX_succ_e",
        "oxygen": True,
        "carbon_uptake": 10
    },
    "acetate_aerobic": {
        "carbon_source": "EX_ac_e",
        "oxygen": True,
        "carbon_uptake": 10
    },
    "glucose_anaerobic": {
        "carbon_source": "EX_glc__D_e",
        "oxygen": False,
        "carbon_uptake": 10
    }
}

conditions

{'glucose_aerobic': {'carbon_source': 'EX_glc__D_e',
  'oxygen': True,
  'carbon_uptake': 10},
 'fructose_aerobic': {'carbon_source': 'EX_fru_e',
  'oxygen': True,
  'carbon_uptake': 10},
 'galactose_aerobic': {'carbon_source': 'EX_gal_e',
  'oxygen': True,
  'carbon_uptake': 10},
 'glycerol_aerobic': {'carbon_source': 'EX_glyc_e',
  'oxygen': True,
  'carbon_uptake': 10},
 'succinate_aerobic': {'carbon_source': 'EX_succ_e',
  'oxygen': True,
  'carbon_uptake': 10},
 'acetate_aerobic': {'carbon_source': 'EX_ac_e',
  'oxygen': True,
  'carbon_uptake': 10},
 'glucose_anaerobic': {'carbon_source': 'EX_glc__D_e',
  'oxygen': False,
  'carbon_uptake': 10}}

## 8. FBA analysis function for model comparison

This function applies one medium condition to one model, runs Flux Balance Analysis, extracts summary metrics, and saves the corresponding flux and blocked reaction tables.

The function is designed to be reused for both iML1515 and iJO1366 under the same set of medium conditions.

In [ ]:
def analyze_model_condition(base_model, model_name, condition_name, carbon_source, oxygen, carbon_uptake=10):
    """
    Apply one medium condition to one model, run FBA,
    and extract comparable summary metrics.

    Blocked reactions are loaded from saved files when available,
    to avoid recalculating them during every Run All execution.
    """

    model_condition = set_medium_from_default(
        base_model=base_model,
        carbon_source=carbon_source,
        oxygen=oxygen,
        carbon_uptake=carbon_uptake
    )

    solution = model_condition.optimize()

    nonzero_fluxes = int((solution.fluxes.abs() > 1e-9).sum())
    zero_fluxes = int((solution.fluxes.abs() <= 1e-9).sum())

    safe_name = f"{model_name}_{condition_name}"
    blocked_reactions_path = tables_dir / f"{safe_name}_blocked_reactions.csv"

    if blocked_reactions_path.exists():
        blocked_df = pd.read_csv(blocked_reactions_path)
        blocked_reactions = blocked_df["blocked_reaction_id"].dropna().tolist()
    else:
        blocked_reactions = find_blocked_reactions(model_condition)

        blocked_df = pd.DataFrame({
            "model_name": model_name,
            "condition": condition_name,
            "blocked_reaction_id": blocked_reactions
        })

        blocked_df.to_csv(blocked_reactions_path, index=False)

    num_blocked = len(blocked_reactions)
    blocked_fraction = num_blocked / len(model_condition.reactions)

    summary = {
        "model_name": model_name,
        "model_id": model_condition.id,
        "condition": condition_name,
        "carbon_source": carbon_source,
        "oxygen_available": oxygen,
        "carbon_uptake": carbon_uptake,
        "solution_status": solution.status,
        "objective_value": solution.objective_value,
        "num_reactions": len(model_condition.reactions),
        "num_metabolites": len(model_condition.metabolites),
        "num_genes": len(model_condition.genes),
        "num_nonzero_fluxes": nonzero_fluxes,
        "num_zero_fluxes": zero_fluxes,
        "num_blocked_reactions": num_blocked,
        "blocked_reaction_fraction": blocked_fraction
    }

    flux_df = solution.fluxes.reset_index()
    flux_df.columns = ["reaction_id", "flux"]
    flux_df["absolute_flux"] = flux_df["flux"].abs()
    flux_df["model_name"] = model_name
    flux_df["condition"] = condition_name
    flux_df = flux_df.sort_values("absolute_flux", ascending=False)

    flux_df.to_csv(tables_dir / f"{safe_name}_fluxes.csv", index=False)

    return summary, flux_df

## 9. Run FBA for all models and conditions

Flux Balance Analysis is performed for every model-condition combination.

The same medium conditions are applied to both iML1515 and iJO1366. This produces a direct comparison of the predicted objective values, active fluxes, and blocked reactions between the two reconstructions.

In [ ]:
all_model_summaries = []
all_model_flux_tables = {}

for model_name, model_obj in models.items():
    for condition_name, params in conditions.items():
        print("=" * 80)
        print("Model:", model_name)
        print("Condition:", condition_name)

        summary, flux_df = analyze_model_condition(
            base_model=model_obj,
            model_name=model_name,
            condition_name=condition_name,
            carbon_source=params["carbon_source"],
            oxygen=params["oxygen"],
            carbon_uptake=params["carbon_uptake"]
        )

        all_model_summaries.append(summary)
        all_model_flux_tables[(model_name, condition_name)] = flux_df

model_comparison_df = pd.DataFrame(all_model_summaries)

model_comparison_df

Model: iML1515
Condition: glucose_aerobic
Model: iML1515
Condition: fructose_aerobic
Model: iML1515
Condition: galactose_aerobic
Model: iML1515
Condition: glycerol_aerobic
Model: iML1515
Condition: succinate_aerobic
Model: iML1515
Condition: acetate_aerobic
Model: iML1515
Condition: glucose_anaerobic
Model: iJO1366
Condition: glucose_aerobic
Model: iJO1366
Condition: fructose_aerobic
Model: iJO1366
Condition: galactose_aerobic


In [ ]:
model_comparison_summary_path = tables_dir / "ecoli_model_comparison_iML1515_iJO1366_summary.csv"

model_comparison_df.to_csv(model_comparison_summary_path, index=False)

print("Saved:", model_comparison_summary_path)

In [ ]:
model_comparison_report_table = model_comparison_df[
    [
        "model_name",
        "condition",
        "carbon_source",
        "oxygen_available",
        "objective_value",
        "num_nonzero_fluxes",
        "num_blocked_reactions",
        "blocked_reaction_fraction"
    ]
].copy()

model_comparison_report_table["objective_value"] = (
    model_comparison_report_table["objective_value"].round(4)
)

model_comparison_report_table["blocked_reaction_fraction"] = (
    model_comparison_report_table["blocked_reaction_fraction"].round(4)
)

model_comparison_report_table

In [ ]:
model_comparison_report_path = tables_dir / "ecoli_model_comparison_iML1515_iJO1366_report_table.csv"

model_comparison_report_table.to_csv(model_comparison_report_path, index=False)

print("Saved:", model_comparison_report_path)

## 10. Plots for model comparison

This section generates bar plots comparing the two E. coli models across the same medium conditions.

The plots show differences in predicted objective value, blocked reactions, and active fluxes between iML1515 and iJO1366.

In [ ]:
objective_pivot = model_comparison_df.pivot(
    index="condition",
    columns="model_name",
    values="objective_value"
)

objective_pivot.plot(kind="bar", figsize=(10, 5))

plt.title("FBA objective value comparison between iML1515 and iJO1366")
plt.xlabel("Medium condition")
plt.ylabel("Objective value")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

figure_path = figures_dir / "ecoli_model_comparison_objective_value.png"
plt.savefig(figure_path, dpi=300)

print("Saved:", figure_path)
plt.show()

In [ ]:
blocked_pivot = model_comparison_df.pivot(
    index="condition",
    columns="model_name",
    values="num_blocked_reactions"
)

blocked_pivot.plot(kind="bar", figsize=(10, 5))

plt.title("Blocked reactions comparison between iML1515 and iJO1366")
plt.xlabel("Medium condition")
plt.ylabel("Number of blocked reactions")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

figure_path = figures_dir / "ecoli_model_comparison_blocked_reactions.png"
plt.savefig(figure_path, dpi=300)

print("Saved:", figure_path)
plt.show()

In [ ]:
active_flux_pivot = model_comparison_df.pivot(
    index="condition",
    columns="model_name",
    values="num_nonzero_fluxes"
)

active_flux_pivot.plot(kind="bar", figsize=(10, 5))

plt.title("Active fluxes comparison between iML1515 and iJO1366")
plt.xlabel("Medium condition")
plt.ylabel("Number of nonzero fluxes")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

figure_path = figures_dir / "ecoli_model_comparison_active_fluxes.png"
plt.savefig(figure_path, dpi=300)

print("Saved:", figure_path)
plt.show()

## 11. Difference tables

This section calculates numerical differences between iML1515 and iJO1366 for each medium condition.

The objective value difference shows how much the predicted growth-related objective changes between models.

The blocked reaction difference shows how much the number of unavailable reactions differs between the two reconstructions.

In [ ]:
objective_difference_df = objective_pivot.copy()

objective_difference_df["absolute_difference"] = (
    objective_difference_df["iJO1366"] - objective_difference_df["iML1515"]
)

objective_difference_df["relative_difference_percent"] = (
    objective_difference_df["absolute_difference"] / objective_difference_df["iML1515"] * 100
)

objective_difference_df = objective_difference_df.reset_index()

objective_difference_df["iJO1366"] = objective_difference_df["iJO1366"].round(4)
objective_difference_df["iML1515"] = objective_difference_df["iML1515"].round(4)
objective_difference_df["absolute_difference"] = objective_difference_df["absolute_difference"].round(4)
objective_difference_df["relative_difference_percent"] = objective_difference_df["relative_difference_percent"].round(2)

objective_difference_df

In [ ]:
objective_difference_path = tables_dir / "ecoli_model_comparison_objective_differences.csv"

objective_difference_df.to_csv(objective_difference_path, index=False)

print("Saved:", objective_difference_path)

In [ ]:
blocked_difference_df = blocked_pivot.copy()

blocked_difference_df["absolute_difference_iML1515_minus_iJO1366"] = (
    blocked_difference_df["iML1515"] - blocked_difference_df["iJO1366"]
)

blocked_difference_df["relative_difference_percent"] = (
    blocked_difference_df["absolute_difference_iML1515_minus_iJO1366"]
    / blocked_difference_df["iJO1366"]
    * 100
)

blocked_difference_df = blocked_difference_df.reset_index()

blocked_difference_df["relative_difference_percent"] = (
    blocked_difference_df["relative_difference_percent"].round(2)
)

blocked_difference_df

In [ ]:
blocked_difference_path = tables_dir / "ecoli_model_comparison_blocked_reaction_differences.csv"

blocked_difference_df.to_csv(blocked_difference_path, index=False)

print("Saved:", blocked_difference_path)

## 12. Computational summary

This notebook compared two genome-scale metabolic models of *Escherichia coli*, iML1515 and iJO1366, under the same seven medium conditions.

The comparison was designed to keep the environmental conditions constant while changing the model reconstruction. This allowed the analysis to focus on differences between the two models rather than differences in medium setup.

Both models produced optimal FBA solutions under all tested conditions. Across all conditions, iJO1366 produced higher objective values than iML1515. However, both models followed a similar overall pattern across medium conditions: aerobic glucose, fructose, and galactose produced the highest objective values, glycerol and succinate produced intermediate values, and acetate and anaerobic glucose produced lower values.

The number of active fluxes was relatively similar between the two models. In contrast, iML1515 consistently showed a higher number of blocked reactions than iJO1366 under the tested conditions.

The largest relative objective difference was observed under the anaerobic glucose condition. This suggests that oxygen availability can amplify differences between metabolic reconstructions.

Overall, the analysis shows that different genome-scale reconstructions of the same organism can produce similar qualitative trends but different quantitative FBA predictions.